In [12]:
%pip install pandas

import pandas as pd
import numpy as np

pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 200)


Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 24.2 -> 25.3
[notice] To update, run: python.exe -m pip install --upgrade pip


In [13]:
# Load datasets
diabetic_df = pd.read_csv("C:/internship/datasets/raw/diabetic_data.csv")
ids_mapping_df = pd.read_csv("C:/internship/datasets/raw/IDS_mapping.csv")

In [14]:
# Shape
print("Diabetic data shape:", diabetic_df.shape)
print("IDS mapping shape:", ids_mapping_df.shape)


Diabetic data shape: (101766, 50)
IDS mapping shape: (67, 2)


In [15]:
# Replace encoded missing values
diabetic_df = diabetic_df.replace("?", np.nan)


In [16]:
# Verify no "?" remains
(diabetic_df == "?").sum()


encounter_id                0
patient_nbr                 0
race                        0
gender                      0
age                         0
weight                      0
admission_type_id           0
discharge_disposition_id    0
admission_source_id         0
time_in_hospital            0
payer_code                  0
medical_specialty           0
num_lab_procedures          0
num_procedures              0
num_medications             0
number_outpatient           0
number_emergency            0
number_inpatient            0
diag_1                      0
diag_2                      0
diag_3                      0
number_diagnoses            0
max_glu_serum               0
A1Cresult                   0
metformin                   0
repaglinide                 0
nateglinide                 0
chlorpropamide              0
glimepiride                 0
acetohexamide               0
glipizide                   0
glyburide                   0
tolbutamide                 0
pioglitazo

In [17]:
missing_audit = (
    diabetic_df.isna()
    .sum()
    .to_frame(name="missing_count")
)

missing_audit["missing_pct"] = (
    missing_audit["missing_count"] / len(diabetic_df) * 100
)

missing_audit = missing_audit.sort_values(
    by="missing_pct", ascending=False
)


In [18]:
missing_audit

,missing_count,missing_pct
weight,98569,96.858479
max_glu_serum,96420,94.746772
A1Cresult,84748,83.277322
medical_specialty,49949,49.082208
payer_code,40256,39.557416
race,2273,2.233555
diag_3,1423,1.398306
diag_2,358,0.351787
diag_1,21,0.020636
patient_nbr,0,0.000000


In [19]:
diabetic_df["readmit_30"] = np.where(
    diabetic_df["readmitted"] == "<30",
    1,
    0
)


In [25]:
diabetic_df["readmit_30"].value_counts(dropna=False)


readmit_30
0    90409
1    11357
Name: count, dtype: int64

In [26]:
# Cross-check against original
pd.crosstab(diabetic_df["readmitted"], diabetic_df["readmit_30"])


readmit_30,0,1
readmitted,,
<30,0,11357
>30,35545,0
NO,54864,0


In [22]:
identifier_columns = [
    "encounter_id",
    "patient_nbr"
]

identifier_columns


['encounter_id', 'patient_nbr']

In [23]:
diabetic_df.info()


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 101766 entries, 0 to 101765
Data columns (total 51 columns):
 #   Column                    Non-Null Count   Dtype 
---  ------                    --------------   ----- 
 0   encounter_id              101766 non-null  int64 
 1   patient_nbr               101766 non-null  int64 
 2   race                      99493 non-null   object
 3   gender                    101766 non-null  object
 4   age                       101766 non-null  object
 5   weight                    3197 non-null    object
 6   admission_type_id         101766 non-null  int64 
 7   discharge_disposition_id  101766 non-null  int64 
 8   admission_source_id       101766 non-null  int64 
 9   time_in_hospital          101766 non-null  int64 
 10  payer_code                61510 non-null   object
 11  medical_specialty         51817 non-null   object
 12  num_lab_procedures        101766 non-null  int64 
 13  num_procedures            101766 non-null  int64 
 14  num_

In [24]:
# Save missing value audit
missing_audit.to_csv(
    "missing_value_audit.csv",
    index=True
)

# Save cleaned base table (still mostly raw)
diabetic_df.to_csv(
    "diabetic_data_base_table.csv",
    index=False
)
